## Setup do Ambiente

- Importação de bibliotecas essenciais do PySpark
- Definição dos paths utilizados para as tabelas
- Utilização do catálogo `catalogo`, schemas `silver_db_name`, `gold_db_name`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    explode,
    sequence,
    col,
    count,
    min,
    round,
    max,
    sum,
    avg,
    lit,
    desc,
    struct,
    concat_ws,
    collect_list,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    when,
    to_date,    
    current_timestamp
)
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql.utils import AnalysisException

#### Definição de Variáveis Globais

- Centralizamos os nomes de catálogos, bancos de dados e caminhos.
- **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.

In [0]:
catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

In [0]:
spark.sql(f"USE CATALOG {catalogo};")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_db_name};")
spark.sql(f"USE SCHEMA {gold_db_name};")

## Funções Úteis

### Função `table_check`

Esta função verifica se uma tabela existe em um banco de dados específico e se ela contém dados.

In [0]:
def table_check(table_name, db_name):
    """
    Verifica se uma tabela existe e possui dados em um banco de dados especificado.

    Args:
        table_name (str): Nome da tabela a ser verificada.
        db_name (str): Nome do banco de dados onde a tabela está localizada.

    Returns:
        bool: True se a tabela existe e possui dados, False caso contrário.

    Raises:
        ValueError: Se a tabela existe mas está vazia.
    """
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):
        if spark.table(f"{db_name}.{table_name}").count() == 0:
            raise ValueError(f"Tabela {db_name}.{table_name} existe mas está vazia.")
        return True
    return False

### Função `save_table_gold` 
Salva uma tabela a partir do catálogo (`catalogo`) e camada (`gold_db_name`) especificados, em formato Delta.
Adiciona a coluna `data_criacao_gold`

In [0]:
def save_table_gold(table_name: str, df, process_col=True):
    """
    Salva um DataFrame como tabela Delta na camada gold.

    Args:
        table_name (str): Nome da tabela a ser criada ou sobrescrita na camada gold.
        df (DataFrame): DataFrame Spark que será salvo como tabela Delta.
        process_col(Bool): Adicionar uma coluna com o current timestamp.
    """
    table_path = f"{catalogo}.{gold_db_name}.{table_name}"

    try:
        old_schema = spark.table(table_path).schema.simpleString()
    except AnalysisException:
        old_schema = None

    if process_col == True:
        df = df.withColumn("data_criacao_gold", current_timestamp())
    
    try:
        df.write \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .mode("overwrite") \
            .saveAsTable(table_path)
    except Exception as ex:
        print(f"Erro ao salvar a tabela {table_path}: {ex}")
        return
    
    new_schema = spark.table(table_path).schema.simpleString()

    if old_schema is None:
        print(f"Tabela {table_path} criada pela primeira vez.")
    elif old_schema != new_schema:
        print(f"Esquema da tabela {table_path} foi alterado.\n")
        print("Schema anterior:")
        print(old_schema)
        print("\nNovo schema:")
        print(new_schema)
    else:
        print(f"Tabela salva com sucesso: {table_path}")

### Função `read_table`
Lê uma tabela do Databricks a partir do catálogo e camada especificados (`silver` ou `gold`), retornando um DataFrame Spark correspondente.

In [0]:
def read_table(nome_tabela: str, camada:str='silver'):
    """
    Lê uma tabela Delta da camada especificada.

    Args:
        nome_tabela (str): Nome da tabela a ser lida.
        camada (str, optional): Camada de origem da tabela ('silver' ou 'gold'). Default é 'bronze'.

    Returns:
        DataFrame: DataFrame Spark da tabela lida.

    Raises:
        ValueError: Se a camada não for 'silver' ou 'gold'.
        ValueError: Se a tabela não existir ou estiver vazia.
    """
    db_map = {
        'silver': silver_db_name,
        'gold': gold_db_name
    }

    db_nome = db_map.get(camada.lower())
    
    if not db_nome:
        raise ValueError("Camada deve ser 'silver' ou 'gold'")
    
    if not table_check(nome_tabela, db_nome):
        raise ValueError(f"Tabela {db_nome}.{nome_tabela} não existe.")
    return spark.table(f"{catalogo}.{db_nome}.{nome_tabela}")

### Função `describe_table`
Exibe o schema, contagem de linhas com valores nulos e 5 linhas da tabela.

In [0]:
def describe_table(df):
    """
    Exibe o schema, linhas com valores nulos e as primeiras 5 linhas do DataFrame usando display.

    Args:
        df (DataFrame): DataFrame Spark a ser exibido.
    """
    df.printSchema()
    print(f"Total de linhas: {df.count()}")
    null_count = df.filter(
        reduce(lambda a, b: a | b, [F.col(c).isNull() for c in df.columns])
    ).count()
    print(f"Linhas com valores nulos: {null_count}")
    display(df.limit(5))

## Criação da Tabela `dm_tempo`

Colunas da tabela `dm_tempo`:

- `sk_tempo`
- `ano`
- `trimestre`
- `mes`
- `semana_do_ano`
- `dia`
- `dia_da_semana_num`
- `dia_da_semana_nome`
- `mes_nome`
- `eh_fim_de_semana`

In [0]:
df_ft_chamados_hora = read_table("ft_chamados_hora", "silver")

min_max_tempo = df_ft_chamados_hora.agg(
    F.min("hora_abertura_chamado").alias("min_tempo"),
    F.max("hora_finalizacao_atendimento").alias("max_tempo")
)

min_data, max_data = min_max_tempo.withColumn("min_data", to_date("min_tempo")) \
    .withColumn("max_data", to_date("max_tempo")) \
    .select("min_data", "max_data") \
    .first()

Datas usadas no dataset: 
- 2025-01-01 à 2025-06-30
- Checamos as datas min e max, passamos como parâmetros para garantir o bom funcionamento com datas futuras.

In [0]:
df_datas = (
    spark.createDataFrame([(min_data, max_data)], ['min_data', 'max_data'])
    .select(explode(sequence(col('min_data').cast(DateType()), col('max_data').cast(DateType()))).alias('sk_tempo'))
    .withColumn('ano', year(col('sk_tempo')))
    .withColumn('trimestre', quarter(col('sk_tempo')))
    .withColumn('mes', month(col('sk_tempo')))
    .withColumn('semana_do_ano', weekofyear(col('sk_tempo')))
    .withColumn('dia', dayofmonth(col('sk_tempo')))
    .withColumn('dia_da_semana_num', dayofweek(col('sk_tempo')))
    .withColumn('dia_da_semana_nome', 
        when(col('dia_da_semana_num') == 1, 'Domingo')
        .when(col('dia_da_semana_num') == 2, 'Segunda-feira')
        .when(col('dia_da_semana_num') == 3, 'Terça-feira')
        .when(col('dia_da_semana_num') == 4, 'Quarta-feira')
        .when(col('dia_da_semana_num') == 5, 'Quinta-feira')
        .when(col('dia_da_semana_num') == 6, 'Sexta-feira')
        .when(col('dia_da_semana_num') == 7, 'Sabado')
    )
    .withColumn('mes_nome',
        when(col('mes') == 1, 'Janeiro')
        .when(col('mes') == 2, 'Fevereiro')
        .when(col('mes') == 3, 'Março')
        .when(col('mes') == 4, 'Abril')
        .when(col('mes') == 5, 'Maio')
        .when(col('mes') == 6, 'Junho')
        .when(col('mes') == 7, 'Julho')
        .when(col('mes') == 8, 'Agosto')
        .when(col('mes') == 9, 'Setembro')
        .when(col('mes') == 10, 'Outubro')
        .when(col('mes') == 11, 'Novembro')
        .when(col('mes') == 12, 'Dezembro')
    )
    .withColumn('eh_fim_de_semana', when(col('dia_da_semana_num').isin([1,7]), 'Sim').otherwise('Não'))
)

describe_table(df_datas)

save_table_gold("dm_tempo", df_datas)

## Criação da tabela `ft_chamados_tempo`

In [0]:
df_ft_chamados_geral = read_table("ft_chamados_geral", "silver")

df_dm_tempo = read_table("dm_tempo", "gold")

df_join = df_ft_chamados_geral.join(
    df_dm_tempo,
    to_date(col("hora_abertura_chamado")) == df_dm_tempo["sk_tempo"],
    "inner"
).select(
    df_dm_tempo["sk_tempo"].alias("data"),
    col("id_chamado"),
    col("id_cliente"),
    col("id_atendente"),
    col("motivo"),
    col("categoria"),
    col("nota_atendimento"),
    col("criticidade"),
    col("canal"),
    col("status_canal"),
    col("resolvido"),
    col("hora_inicio_atendimento"),
    col("tempo_espera_segundos"),
    col("tempo_atendimento_segundos"),
    col("nome_atendente"),
    col("nivel_atendimento"),
    col("valor_custo"),
    col("regiao"),
    col("idade"),
    df_dm_tempo["trimestre"],
    df_dm_tempo["mes_nome"],
    df_dm_tempo["dia_da_semana_nome"],
    df_dm_tempo["eh_fim_de_semana"]
).orderBy("data")

save_table_gold("ft_chamados_tempo", df_join)

describe_table(df_join)

### Criação da view `vw_chamados_tempo`

In [0]:
df_ft_chamados_tempo = read_table("ft_chamados_tempo", "gold")

spark.sql(f"""
CREATE OR REPLACE VIEW {catalogo}.{gold_db_name}.vw_chamados_tempo AS
SELECT
    id_cliente,
    motivo,
    categoria,
    canal,
    status_canal,
    resolvido,
    nivel_atendimento,
    valor_custo,
    regiao,
    idade,
    dia_da_semana_nome,
    eh_fim_de_semana,
    hora_inicio_atendimento,
    tempo_atendimento_segundos,
    tempo_espera_segundos
FROM {catalogo}.{gold_db_name}.ft_chamados_tempo
""")

## Dores na Visão do Gerente Financeiro



- Não encontramo diferenças significativas de custo por faixa etária ou região atendida.
- Diferenças significativas entre os canais utilizados.

In [0]:
# Custo médio por faixa etária (intervalos de 10 anos a partir de 18)
df_media_custo_faixa_idade = df_ft_chamados_tempo.withColumn(
    "faixa_etaria",
    when(col("idade") < 18, "Menor de idade")
    .when((col("idade") >= 18) & (col("idade") < 28), "18-27")
    .when((col("idade") >= 28) & (col("idade") < 38), "28-37")
    .when((col("idade") >= 38) & (col("idade") < 48), "38-47")
    .when((col("idade") >= 48) & (col("idade") < 58), "48-57")
    .when((col("idade") >= 58) & (col("idade") < 68), "58-67")
    .otherwise("68+")
).groupBy("faixa_etaria").agg(
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
).orderBy("faixa_etaria")
display(df_media_custo_faixa_idade)

# Custo médio por canal
df_media_custo_atendimento = df_ft_chamados_tempo.groupBy("canal").agg(
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
)
display(df_media_custo_atendimento)

# Custo médio por nível de atendimento
df_media_custo_nivel = df_ft_chamados_tempo.groupBy("nivel_atendimento").agg(
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
)
display(df_media_custo_nivel)

# Custo médio por região atendida
df_media_custo_regiao = df_ft_chamados_tempo.groupBy("regiao").agg(
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
)
display(df_media_custo_regiao)

### View `vw_media_custo_minuto`

In [0]:

spark.sql(f"""
CREATE OR REPLACE VIEW {catalogo}.{gold_db_name}.vw_media_custo_minuto AS
SELECT
    make_timestamp(2025, 1, 1, hour(hora_inicio_atendimento), minute(hora_inicio_atendimento), 0) AS ts_hora_minuto,
    hora_inicio_atendimento,
    ROUND(AVG(valor_custo), 2) AS media_custo,
    ROUND(AVG(tempo_espera_segundos), 2) AS media_tempo_espera_segundos,
    ROUND(AVG(tempo_atendimento_segundos), 2) AS media_tempo_atendimento_segundos
FROM {catalogo}.{gold_db_name}.ft_chamados_tempo
GROUP BY ts_hora_minuto, hora_inicio_atendimento
ORDER BY ts_hora_minuto, hora_inicio_atendimento
""")

## Consultas por tempo utilizando `ft_chamados_tempo`

In [0]:
# Custo médio por dia da semana
df_media_custo_dia_semana = df_ft_chamados_tempo.groupBy("dia_da_semana_nome").agg(
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
).orderBy("media_custo")
display(df_media_custo_dia_semana)

# Custo médio por horário do dia
df_media_custo_hora = df_ft_chamados_tempo.withColumn(
    "hora_do_dia", F.hour(col("hora_inicio_atendimento"))
).groupBy("hora_do_dia").agg(
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
).orderBy("media_custo")
display(df_media_custo_hora)

# Custo médio por minuto do dia
df_media_custo_minuto = df_ft_chamados_tempo.withColumn(
    "minuto_do_dia", F.minute(col("hora_inicio_atendimento"))
).withColumn(
    "hora_do_dia", F.hour(col("hora_inicio_atendimento"))
).groupBy("hora_do_dia", "minuto_do_dia").agg(
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
).withColumn("hora_do_dia", col("hora_do_dia")) \
 .orderBy("hora_do_dia", "minuto_do_dia")
display(df_media_custo_minuto)

### Consultas em `ft_chamados_tempo`

In [0]:
df_funil_chamados = df_ft_chamados_tempo.select(
    col("data"),
    col("id_chamado"),
    col("id_cliente"),
    col("canal"),
    col("status_canal"),
    col("resolvido"),
    col("tempo_espera_segundos"),
    col("tempo_atendimento_segundos"),
    col("hora_inicio_atendimento"),
    col("valor_custo"),
    col("regiao"),
    col("idade"),
    col("dia_da_semana_nome"),
    col("eh_fim_de_semana")
)

# add coluna numero de chamados por cliente
window_cliente = Window.partitionBy("id_cliente").orderBy("data")
df_chamados_por_cliente = df_funil_chamados.withColumn(
    "num_chamado_cliente",
    F.row_number().over(window_cliente)
).orderBy("id_cliente", "num_chamado_cliente")

describe_table(df_chamados_por_cliente)

In [0]:
# chamados_por_cliente - consulta
df_chamados_por_cliente = df_ft_chamados_tempo.groupBy("id_cliente").agg(
    F.count("id_chamado").alias("num_chamados")
).orderBy(F.desc("num_chamados"))

describe_table(df_chamados_por_cliente)

# Custo médio por cliente por quantidade de chamados - consulta
df_media_custo_por_num_chamados = df_ft_chamados_tempo.groupBy("id_cliente").agg(
    F.count("id_chamado").alias("num_chamados"),
    F.round(F.avg("valor_custo"), 2).alias("media_custo")
).groupBy("num_chamados").agg(
    F.round(F.avg("media_custo"), 2).alias("media_custo_por_cliente"),
    F.count("id_cliente").alias("qtd_clientes")
).orderBy("num_chamados")

display(df_media_custo_por_num_chamados)

In [0]:
# canal por num_chamados por cliente - consulta
df_canal_por_num_chamados = (
    df_ft_chamados_tempo.groupBy("id_cliente", "canal")
    .agg(F.count("id_chamado").alias("qtd_uso_canal"))
    .join(
        df_chamados_por_cliente.select("id_cliente", "num_chamados"),
        on="id_cliente"
    )
    .groupBy("num_chamados", "canal")
    .agg(
        F.round(F.avg("qtd_uso_canal"), 2).alias("media_uso_canal"),
        F.sum("qtd_uso_canal").alias("total_uso_canal")
    )
    .withColumn(
        "pct_uso_canal",
        F.round(
            col("total_uso_canal") / F.sum("total_uso_canal").over(Window.partitionBy("num_chamados")) * 100, 2
        )
    )
    .orderBy("num_chamados", "canal")
)

display(df_canal_por_num_chamados)

In [0]:
# motivos por numero de chamado - consulta

df_motivo_por_num_chamados = (
    df_ft_chamados_tempo.groupBy("id_cliente", "motivo")
    .agg(F.count("id_chamado").alias("qtd_chamados"))
    .join(
        df_chamados_por_cliente.select("id_cliente", "num_chamados"),
        on="id_cliente"
    )
    .groupBy("num_chamados", "motivo")
    .agg(
        F.count("id_cliente").alias("qtd_clientes_motivo")
    )
    .withColumn(
        "pct_clientes_motivo",
        F.round(
            col("qtd_clientes_motivo") / F.sum("qtd_clientes_motivo").over(Window.partitionBy("num_chamados")) * 100, 2
        )
    )
    .orderBy("num_chamados", F.desc("pct_clientes_motivo"))
)

display(df_motivo_por_num_chamados)

In [0]:
# chamados resolvidos por motivo e canal - consulta

df_resolucao_motivo_canal = (
    df_ft_chamados_tempo.filter(col("resolvido") == "Sim")
    .groupBy("motivo", "canal")
    .agg(F.count("id_chamado").alias("qtd_resolvidos"))
    .withColumn(
        "pct_resolvidos_motivo",
        F.round(
            col("qtd_resolvidos") / F.sum("qtd_resolvidos").over(Window.partitionBy("motivo")) * 100, 2
        )
    )
    .orderBy("motivo", "canal")
)

display(df_resolucao_motivo_canal)

## Tabela `ft_chamados_por_cliente`

In [0]:
df_funil_chamados = df_ft_chamados_tempo.select(
    col("data"),
    col("id_chamado"),
    col("id_cliente"),
    col("canal"),
    col("status_canal"),
    col("resolvido"),
    col("tempo_espera_segundos"),
    col("tempo_atendimento_segundos"),
    col("hora_inicio_atendimento"),
    col("valor_custo"),
    col("regiao"),
    col("idade"),
    col("dia_da_semana_nome"),
    col("eh_fim_de_semana")
)

# add coluna numero de chamados por cliente
window_cliente = Window.partitionBy("id_cliente").orderBy("data")
df_chamados_por_cliente = df_funil_chamados.withColumn(
    "num_chamado_cliente",
    F.row_number().over(window_cliente)
).orderBy("id_cliente", "num_chamado_cliente")

save_table_gold("ft_chamados_por_cliente", df_chamados_por_cliente)

## View `vw_chamados_por_cliente`

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalogo}.{gold_db_name}.vw_chamados_por_cliente AS
SELECT * FROM ft_chamados_por_cliente
""")

## Criação da Tabela `ft_chamados`

Colunas da tabela `ft_chamados`:

- `id_chamado`
- `id_cliente`
- `id_atendente`
- `motivo`
- `canal`
- `resolvido`
- `nota_atendimento`
- `categoria_nota`
- `status_canal`
- `valor_custo`

In [0]:
df_chamados_geral = spark.table(f'{catalogo}.{silver_db_name}.ft_chamados_geral')
display(df_chamados_geral.limit(20))

In [0]:
print(f"Colunas de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
print(f"{df_chamados_geral.columns}\n")
print(f"Schema de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
df_chamados_geral.printSchema()

In [0]:
df_gold_ft_chamados = df_chamados_geral.select(
    col('id_chamado'),
    col('id_cliente'),
    when(col('id_atendente') == -1, None).otherwise(col('id_atendente')).alias('id_atendente'),
    when(col('nome_atendente').isNull(), "Atendimento Não Humano").otherwise(col('nome_atendente')).alias('nome_atendente'),
    col('motivo'),
    col('hora_inicio_atendimento'),
    col('hora_finalizacao_atendimento'),
    col('canal'),
    col('status_canal'),
    col('resolvido'),
    col('nota_atendimento'),
    col('categoria_nota'),
    col('valor_custo')
)
df_gold_ft_chamados.limit(15).display()
df_gold_ft_chamados.printSchema()

In [0]:
df_tratado = df_gold_ft_chamados.withColumn("ts_inicio", col("hora_inicio_atendimento")) \
                   .withColumn("ts_fim", col("hora_finalizacao_atendimento")) 

GAP_LIMIT_SECONDS = 7 * 24 * 60 * 60  # 7 dias

w_cliente_motivo = Window.partitionBy("id_cliente", "motivo").orderBy(col("ts_inicio"))

Cria uma janela de particionamento para análise temporal
- Agrupa por: cliente + motivo do chamado
- Ordena por: data/hora de início (do mais antigo ao mais recente).Isso permite comparar chamados consecutivos do mesmo cliente sobre o mesmo assunto

In [0]:
df_sessao = df_tratado.withColumn(
    "ts_fim_anterior", F.lag("ts_fim").over(w_cliente_motivo)
).withColumn(
    "segundos_desde_ultimo_contato", 
    col("ts_inicio").cast("long") - col("ts_fim_anterior").cast("long")
).withColumn(
    "nova_jornada_flag",
    when(
        (col("segundos_desde_ultimo_contato").isNull()) | 
        (col("segundos_desde_ultimo_contato") > GAP_LIMIT_SECONDS), 
        1
    ).otherwise(0)
)
df_sessao.filter(col("canal") == "Chatbot").limit(10).display()

- Busca o timestamp de finalização do chamado ANTERIOR
-   Usa lag() para "olhar para trás" na janela ordenada
-   Retorna NULL na primeira linha de cada partição (não há anterior)
-   Subtrai: (quando este chamado começou) - (quando o anterior terminou)
-   Conversão para "long" 
-    Resultado NULL indica que é o primeiro chamado da partição
-  Flag binária que marca o início de uma nova jornada
-   Valor 1: É uma nova jornada (primeiro contato OU gap > 7 dias)
-   Valor 0: Continua na mesma jornada (gap <= 7 dias)

In [0]:
ft_jornada_atendimento = df_sessao.groupBy("id_cliente", "motivo").agg(
    
    min("ts_inicio").alias("data_inicio_jornada"),
    max("ts_fim").alias("data_fim_jornada"),
    count("id_chamado").alias("qtd_tentativas"),
    
    round(
        (max("ts_fim").cast("long") - min("ts_inicio").cast("long")) / 1000 / 60, 2
    ).alias("duracao_jornada_minutos"),

    round(sum("valor_custo"), 2).alias("custo_total_jornada"),
    
    max(when(col("id_atendente") != -1, 1).otherwise(0)).alias("flg_passou_humano"),
    
    # Pega o status e categoria_nota do chamado mais recente (maior data de inicio)
    max(struct("ts_inicio", "resolvido"))["resolvido"].alias("status_final_resolucao"),
    max(struct("ts_inicio", "categoria_nota"))["categoria_nota"].alias("satisfacao_cliente"),
    
    concat_ws(" -> ", collect_list("canal")).alias("caminho_canais")
)

ft_jornada_atendimento.orderBy("data_inicio_jornada").limit(10).display()

In [0]:
df_gold_ft_chamados.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.ft_chamados')
ft_jornada_atendimento.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.ft_jornada_atendimento')

### Views

In [0]:
df_jornada = spark.read.table('ft_jornada_atendimento')
df_jornada.printSchema()

#### View Operacional por Tipo de Atendimento
- **Objetivo**:  comparar jornadas que passaram por atendimento humano versus jornadas atendidas apenas por bot, divididas por status final de resolução.
- Passos: 
  -  Cria uma coluna categórica baseada na flag `flg_passou_humano`
  - Agrupa por tipo de atendimento e status de resolução
- Resultado:
  - `total_tipo:` Total de jornadas dentro de cada tipo de atendimento
  -` perc_status:` Percentual que cada status representa dentro do seu tipo
| Métrica | Descrição | Fórmula |
|---------|-----------|---------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) |
| `custo_total` | Custo acumulado | SUM(custo_total_jornada) |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) |
| `tentativas_max` | Máximo de tentativas | MAX(qtd_tentativas) |


In [0]:
df_operacional = df_jornada \
    .withColumn("tipo_atendimento", 
                when(col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .groupBy("tipo_atendimento", "status_final_resolucao") \
    .agg(
        count("*").alias("qtd_jornadas"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(sum("custo_total_jornada"), 2).alias("custo_total"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round(max("qtd_tentativas"), 2).alias("tentativas_media")
    ) \
    .withColumn("total_tipo", sum("qtd_jornadas").over(Window.partitionBy("tipo_atendimento"))) \
    .withColumn("perc_status", round((col("qtd_jornadas") / col("total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .withColumn("custo_total_tipo", round(sum("custo_total").over(Window.partitionBy("tipo_atendimento")), 2)) \
    .withColumn("perc_custo", round((col("custo_total") / col("custo_total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("tipo_atendimento", desc("qtd_jornadas")) \
    .withColumn("qtd_resolvidos_tipo", 
                sum(when(col("status_final_resolucao") == "Sim", col("qtd_jornadas")).otherwise(0))
                 .over(Window.partitionBy("tipo_atendimento"))) \
    .withColumn("taxa_resolucao_tipo", 
                round((col("qtd_resolvidos_tipo") / col("total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("tipo_atendimento", desc("qtd_jornadas"))

df_operacional.display()

#### Análise Detalhada por Status de Resolução e Motivo

- **Objetivo:**
Analisar a distribuição de jornadas por **status de resolução** e **motivo de contato**, permitindo identificar quais motivos têm maior taxa de resolução, custo e necessidade de intervenção humana.
- **Passos:**
Agrupar por status de resolução e motivos.

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) | Volume absoluto dessa combinação |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) | Quanto custa em média essa jornada |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Quanto tempo demora para resolver |
| `tentativas_media` | Média de tentativas | AVG(qtd_tentativas) | Quantas vezes o cliente precisou tentar |
| `perc_passou_humano` | % com atendimento humano | (SUM(flg_passou_humano) / COUNT(*)) × 100 | Taxa de escalonamento para humano |
| `perc_dentro_status` | %  cada motivo representa dentro do seu status | Window function soma todas as jornadas do mesmo status | "Do total de jornadas resolvidas, 30% foram sobre Boleto"|
| `perc_total` | Calcula quanto cada combinação status+motivo representa do total geral|  Window.partitionBy(F.lit(1)) cria uma janela sobre toda a tabela | "Esta combinação representa 6% de todas as jornadas" |


In [0]:
df_status_motivo = df_jornada \
    .groupBy("status_final_resolucao", "motivo") \
    .agg(
        count("*").alias("qtd_jornadas"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round(avg("qtd_tentativas"), 2).alias("tentativas_media"),
        round((sum("flg_passou_humano") / count("*") * 100).cast("decimal(5,2)"), 2).alias("perc_passou_humano")
    ) \
    .withColumn("perc_dentro_status", round((col("qtd_jornadas") /  sum("qtd_jornadas").over(Window.partitionBy("status_final_resolucao"))* 100).cast("decimal(5,2)"), 2)) \
    .withColumn("perc_total", 
    round((col("qtd_jornadas") / 
             sum("qtd_jornadas").over(Window.partitionBy(lit(1))) * 100)
            .cast("decimal(5,2)"), 2)) \
    .orderBy("status_final_resolucao", desc("qtd_jornadas"))

df_status_motivo.display()

#### Análise de Satisfação do Cliente por Motivo

- **Objetivo:**
Analisar como os clientes avaliam sua experiência em cada **motivo de contato**, correlacionando satisfação com métricas operacionais como custo, duração e tipo de atendimento.

- **Passos:**
1. Filtrar apenas jornadas onde o cliente avaliou (`satisfacao_cliente IS NOT NULL`)
2. Agrupar por motivo e satisfação do cliente
3. Calcular métricas operacionais por combinação motivo + satisfação
4. Adicionar contexto percentual dentro de cada motivo

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_avaliacoes` | Total de avaliações | COUNT(*) | Volume de feedback nessa combinação |
| `custo_medio` | Custo médio | AVG(custo_total_jornada) | Correlacionar custo × satisfação |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Ver se tempo impacta satisfação |
| `perc_passou_humano` | % com atendimento humano | (SUM(flg_passou_humano) / COUNT(*)) × 100 | Verificar se humano melhora satisfação |
| `total_motivo` | Total de avaliações do motivo | Window function soma todas avaliações do mesmo motivo | Base para cálculo percentual |
| `perc_satisfacao` | % que cada nota representa no motivo | (qtd_avaliacoes / total_motivo) × 100 | "70% dos clientes que avaliaram Boleto ficaram satisfeitos" |


In [0]:
df_satisfacao_motivo = df_jornada \
    .filter(col("satisfacao_cliente").isNotNull()) \
    .groupBy("motivo", "satisfacao_cliente") \
    .agg(
        count("*").alias("qtd_avaliacoes"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round((sum("flg_passou_humano") / count("*") * 100).cast("decimal(5,2)"), 2).alias("perc_passou_humano")
    ) \
    .withColumn("total_motivo", sum("qtd_avaliacoes").over(Window.partitionBy("motivo"))) \
    .withColumn("perc_satisfacao", round((col("qtd_avaliacoes") / col("total_motivo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("motivo", "satisfacao_cliente")

df_satisfacao_motivo.display()

#### Análise de Eficiência por Tipo de Atendimento

- **Objetivo:**
Comparar a **eficiência operacional** de jornadas atendidas apenas por bot versus jornadas que passaram por atendimento humano, segmentadas por motivo de contato.

- **Passos:**
1. Agrupar por motivo e flag de atendimento humano
2. Calcular métricas de performance (duração, custo, tentativas)
3. Calcular custo por minuto para medir eficiência
4. Classificar tipo de atendimento para melhor legibilidade
5. Ordenar por motivo e tipo para facilitar comparação

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) | Volume em cada tipo de atendimento |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Quanto tempo demora para resolver |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) | Quanto custa em média essa jornada |
| `tentativas_media` | Média de tentativas | AVG(qtd_tentativas) | Quantas vezes o cliente precisou tentar |
| `tipo_atendimento` | Classificação do atendimento | "Humano + Bot" ou "Apenas Bot" | Facilita leitura e análise |

In [0]:
df_eficiencia = df_jornada \
    .groupBy("motivo", "flg_passou_humano") \
    .agg(
        count("*").alias("qtd_jornadas"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        round(avg("custo_total_jornada"), 2).alias("custo_medio"),
        round(avg("qtd_tentativas"), 2).alias("tentativas_media")
    ) \
    .withColumn("tipo_atendimento", 
                when(col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .select("motivo", "tipo_atendimento", "qtd_jornadas", "duracao_media_min", "custo_medio", "tentativas_media") \
    .orderBy("motivo", "tipo_atendimento")

df_eficiencia.display()

#### Análise de Satisfação por Tipo de Atendimento

- **Objetivo:**
Mapear a distribuição das **notas de satisfação** dos clientes, comparando a experiência em canais 100% digitais ("Apenas Bot") versus interações híbridas ("Humano + Bot"), detalhado por motivo de contato.

- **Passos:**
1. Filtrar registros para garantir que apenas jornadas avaliadas sejam processadas.
2. Agrupar os dados por motivo, tipo de atendimento e nota atribuída.
3. Calcular métricas absolutas (quantidade de notas e duração média).
4. Utilizar **Window Function** para calcular o total de avaliações por grupo, permitindo o cálculo percentual de cada nota (share).
5. Rotular o tipo de atendimento e ordenar para análise visual.

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `satisfacao_cliente` | Nota da avaliação | Coluna original | A nota dada pelo cliente (ex: CSAT ou NPS) |
| `qtd_avaliacoes` | Volume de notas | COUNT(*) | Quantas pessoas deram especificamente essa nota |
| `perc_satisfacao` | % Representatividade | (Qtd Nota / Total do Tipo) * 100 | Qual a porcentagem de clientes que deram essa nota dentro desse canal |
| `duracao_media_min` | Tempo médio | AVG(duracao_jornada_minutos) | Relação entre o tempo de atendimento e a satisfação |
| `tipo_atendimento` | Canal utilizado | Case When... | Segmentação entre Bot Puro ou Transbordo Humano |

In [0]:
window_spec = Window.partitionBy("motivo", "flg_passou_humano")

df_eficiencia_satisfacao = df_jornada \
    .filter(col("satisfacao_cliente").isNotNull()) \
    .groupBy("motivo", "flg_passou_humano", "satisfacao_cliente") \
    .agg(
        count("*").alias("qtd_avaliacoes"),
        round(avg("duracao_jornada_minutos"), 2).alias("duracao_media_min")
    ) \
    .withColumn("total_por_tipo", F.sum("qtd_avaliacoes").over(window_spec)) \
    .withColumn("perc_satisfacao", 
                round((col("qtd_avaliacoes") / col("total_por_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .withColumn("tipo_atendimento", 
                when(col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .select("motivo", "tipo_atendimento", "satisfacao_cliente", "qtd_avaliacoes", "perc_satisfacao", "duracao_media_min") \
    .orderBy("motivo", "tipo_atendimento", "satisfacao_cliente")

df_eficiencia_satisfacao.display()

In [0]:
df_operacional.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_operacional')
df_status_motivo.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_status_motivo')
df_satisfacao_motivo.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_satisfacao_motivo')
df_eficiencia.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_eficiencia')
df_eficiencia_satisfacao.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{gold_db_name}.mview_eficiencia_satisfacao')

# Criação da tabela ft_clientes

## Releitura e Garantia de Contexto
Para assegurar a disponibilidade dos dados na sessão atual e mitigar eventuais perdas de referência em memória, realizamos a leitura explícita das tabelas `ft_chamados_geral` e `ft_clientes`.
Essa ação garantiu que os DataFrames estivessem atualizados e prontos para o cruzamento final, prevenindo erros de execução nas células subsequentes.

In [0]:
df_chamados_silver = spark.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral")

In [0]:
df_clientes_silver = spark.table(f"{catalogo}.{silver_db_name}.ft_clientes")

## Cálculo de Métricas Comportamentais do Cliente

Nesta etapa, consolidamos o histórico transacional de cada cliente em métricas de comportamento agregadas. Agrupamos os dados da tabela `df_chamados_silver` pela chave `id_cliente` e calculamos seis indicadores fundamentais para compor o perfil do consumidor:

1.  **Volumetria:** Contagem total de chamados realizados.
2.  **Financeiro:** Soma do custo operacional gerado pelo cliente.
3.  **Satisfação:** Média das notas de atendimento atribuídas.
4.  **Esforço:** Tempo total acumulado que o cliente passou em espera.
5.  **Recência:** Identificação da data do último contato para análises de retenção.
6.  **Resolutividade:** Cálculo percentual da taxa de sucesso nas solicitações do cliente.

Essa agregação transforma dados granulares de chamados em uma visão única de "valor e comportamento" por cliente.

In [0]:
# "Fatos" do cliente
df_metrics_comportamento = (
    df_chamados_silver
    .groupBy("id_cliente")
    .agg(
        # 1. Volumetria
        F.count("id_chamado").alias("total_chamados_historico"),
        
        # 2. Financeiro (Custo que o cliente gerou para a operação)
        F.round(F.sum("valor_custo"), 2).alias("custo_total_atendimento"),
        
        # 3. Satisfação (Média das notas dele)
        F.round(F.avg("nota_atendimento"), 1).alias("nota_media_satisfacao"),
        
        # 4. Tempo (Quanto tempo ele já perdeu com a gente)
        F.sum("tempo_espera_segundos").alias("tempo_total_espera_seg"),
        
        # 5. Recência (Quando foi a última vez que ele ligou?)
        F.max("hora_abertura_chamado").alias("data_ultimo_contato"),
        
        # 6. Resolutividade (Quantos % dos problemas dele foram resolvidos)
        F.round(
            (F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)) / F.count("id_chamado")) * 100, 
            2
        ).alias("taxa_resolucao_pessoal")
    )
)

display(df_metrics_comportamento.limit(5))

## Enriquecimento e Segmentação Demográfica do Cliente

Nesta etapa, isolamos os dados cadastrais essenciais da tabela `df_clientes_silver` (como nome, e-mail e região) para compor a dimensão de perfil.

Para potenciar as análises de Marketing na camada Gold, enriquecemos esta base criando a coluna `faixa_etaria`. Aplicamos regras de negócio condicionais para segmentar os clientes em quatro grandes grupos ("Jovem", "Adulto Jovem", "Adulto" e "Senior"), transformando a variável numérica contínua (`idade`) numa variável categórica pronta para filtros e agrupamentos em *dashboards*.

In [0]:
# Tratamento da Dimensão Cliente (Dados Cadastrais)
df_perfil_cliente = (
    df_clientes_silver
    .select("id_cliente", "nome_cliente", "email_cliente", "regiao", "idade")
    
    # Criando Faixa Etária (Facilita análise de Marketing)
    .withColumn("faixa_etaria", 
                F.when(F.col("idade") < 25, "Jovem (ate 24)")
                .when(F.col("idade").between(25, 40), "Adulto Jovem (25-40)")
                .when(F.col("idade").between(41, 60), "Adulto (41-60)")
                .otherwise("Senior (+60)")
    )
)

display(df_perfil_cliente.limit(5))

## Consolidação da Visão 360º do Cliente

Nesta etapa crítica, nós unificamos as dimensões de perfil e comportamento para criar a tabela final. Realizamos um *Left Join* entre a base de clientes e as métricas calculadas, garantindo que mesmo os clientes "silenciosos" (sem histórico de chamados) fossem mantidos na análise.

Para sanear os dados, aplicamos a imputação de zero nos valores nulos resultantes da junção (para métricas de volume e custo). Em seguida, implementamos uma lógica de segmentação estratégica na coluna `perfil_cliente`, classificando os consumidores em categorias como "Detrator Crítico", "Promotor" ou "Neutro" com base na combinação da volumetria de contato e da satisfação média. Por fim, selecionamos as colunas definitivas e adicionamos a data de processamento.

In [0]:
df_gold_cliente_360 = (
    df_perfil_cliente.alias("cli")
    # Left Join: todos os clientes, mesmo os que nunca ligaram
    .join(df_metrics_comportamento.alias("fat"), "id_cliente", "left")
    
    # Tratamento de Nulos 
    .fillna(0, subset=["total_chamados_historico", "custo_total_atendimento", "tempo_total_espera_seg"])
    
    .withColumn("perfil_cliente",
                F.when((F.col("total_chamados_historico") > 5) & (F.col("nota_media_satisfacao") < 5), "DETRATOR CRITICO")
                .when((F.col("nota_media_satisfacao") >= 9), "PROMOTOR")
                .when(F.col("total_chamados_historico") == 0, "SILENCIOSO (SEM CONTATO)")
                .otherwise("NEUTRO / ATIVO")
    )
    
    .select(
        "id_cliente",
        "nome_cliente",
        "email_cliente",
        "regiao",
        "idade",
        "faixa_etaria",
        "total_chamados_historico",
        "custo_total_atendimento",
        "nota_media_satisfacao",
        "taxa_resolucao_pessoal",
        "data_ultimo_contato",
        "perfil_cliente",
        F.current_timestamp().alias("data_processamento_gold")
    )
)

display(df_gold_cliente_360)

## Persistência da Tabela Fato de Perfil do Cliente

Como etapa conclusiva deste fluxo, nós materializamos o DataFrame consolidado `df_gold_cliente_360` no armazenamento físico. Definimos o destino como `ft_cliente_perfil` dentro do esquema Gold.

Utilizamos o formato Delta e configuramos o modo de escrita como `overwrite` (sobrescrita), juntamente com a opção `overwriteSchema`. Essa decisão assegura que a tabela reflita sempre a versão mais atualizada e completa do perfil dos clientes, permitindo inclusive alterações na estrutura dos dados (evolução de esquema) sem causar falhas no processo.

In [0]:
tabela_destino = f"{catalogo}.{gold_db_name}.ft_clientes"

(
    df_gold_cliente_360.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"Tabela Gold (Customer 360) criada com sucesso em: {tabela_destino}")

## Faixa etária dos clientes X etapa em que resolveram o problema

## Leitura dos Dados para Análise de Funil

Para dar início à construção da visão estratégica de "Faixa Etária x Etapa de Resolução", nós carregamos a tabela `ft_chamados_geral` do catálogo. Essa leitura garante que temos em memória o histórico completo e granular dos atendimentos, o que é indispensável para classificar corretamente cada interação segundo o canal utilizado e o perfil do cliente nas etapas seguintes.

In [0]:
df_chamados_geral = spark.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral")

## Modelagem de Etapas do Funil e Demografia

Nesta transformação, nós recriamos a categorização de `faixa_etaria` para assegurar a consistência dos dados demográficos na análise. Simultaneamente, introduzimos uma nova dimensão estratégica chamada `etapa_funil`.

Mapeamos os canais de atendimento numa hierarquia lógica de resolução:
* **Autosserviço:** Agrupamos canais digitais e automatizados (Chatbot, URA, Web, App).
* **Nível 1:** Atendimento humano generalista (Atendimento Inicial).
* **Nível 2:** Atendimento humano especialista (Atendimento Especializado).

Essa estruturação é fundamental para analisarmos posteriormente a eficiência da retenção digital versus o transbordo para o atendimento humano.

In [0]:
df_transformado = (
    df_chamados_geral
    
    # 1. Recriar Faixa Etária (Caso não tenha trazido do join anterior)
    .withColumn("faixa_etaria", 
                F.when(F.col("idade") < 25, "1. Jovem (ate 24)")
                .when(F.col("idade").between(25, 40), "2. Adulto Jovem (25-40)")
                .when(F.col("idade").between(41, 60), "3. Adulto (41-60)")
                .when(F.col("idade") > 60, "4. Senior (+60)")
                .otherwise("5. Nao Informado")
    )
    
    # 2. Definir Etapa do Funil (Onde o chamado parou)
    .withColumn("etapa_funil",
                F.when(F.col("canal").isin("Chatbot", "URA", "Web", "App"), "1. Autosservico (Digital/Robo)")
                .when(F.col("canal") == "Atendimento Inicial", "2. Nivel 1 (Humano Generalista)")
                .when(F.col("canal") == "Atendimento Especializado", "3. Nivel 2 (Especialista)")
                .otherwise("4. Outros")
    )
)

display(df_transformado.select("id_chamado", "canal", "etapa_funil", "idade", "faixa_etaria").limit(5))

## Cálculo de Eficiência do Funil por Faixa Etária

Nesta etapa analítica, nós consolidamos os dados agrupando-os por duas dimensões chave: `faixa_etaria` e `etapa_funil`.

Dentro da agregação, calculamos KPIs fundamentais para entender a performance do atendimento:
* **Volumetria:** O total de tentativas de contato.
* **Sucesso:** O volume absoluto de chamados resolvidos.
* **Esforço:** O Tempo Médio de Atendimento (TMA) em segundos.
* **Qualidade:** A nota média de satisfação atribuída.

Além disso, derivamos matematicamente a `taxa_resolucao_pct`. Essa métrica é crucial para comparar a eficácia relativa de cada canal para cada perfil demográfico, permitindo identificar, por exemplo, se determinados grupos têm maior dificuldade com canais digitais. Por fim, ordenamos o resultado para estruturar a visualização lógica do funil.

In [0]:
df_view_funil_idade = (
    df_transformado
    .groupBy("faixa_etaria", "etapa_funil")
    .agg(
        # Volumetria Total
        F.count("id_chamado").alias("total_tentativas"),
        
        # Quantos foram resolvidos?
        F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)).alias("total_resolvidos"),
        
        # Tempo Médio gasto nessa etapa
        F.round(F.avg("tempo_atendimento_segundos"), 0).alias("tma_medio_seg"),
        
        # Nota média dada nessa etapa
        F.round(F.avg("nota_atendimento"), 2).alias("nota_media")
    )
    
    # Cálculo da Taxa de Resolução (Eficácia do Funil)
    .withColumn("taxa_resolucao_pct", 
                F.round((F.col("total_resolvidos") / F.col("total_tentativas")) * 100, 2))
    
    # Ordenação para o gráfico (Do Jovem pro Senior, do Topo pro Fundo)
    .orderBy("faixa_etaria", "etapa_funil")
)

display(df_view_funil_idade)

In [0]:
tabela_destino = f"{catalogo}.{gold_db_name}.funil_resolucao_etaria"

(
    df_view_funil_idade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"View Gold criada com sucesso: {tabela_destino}")

## Leitura para Análise de Esforço do Cliente

Para investigar a raiz da insatisfação relacionada ao tempo perdido em fila, nós iniciamos a leitura da tabela `ft_chamados_geral`. O objetivo é isolar as métricas de tempo (`tempo_espera_segundos`) e cruzá-las com as dimensões de canal e perfil demográfico, permitindo identificar onde estão os maiores gargalos de atendimento.

## Cálculo da Jornada de Espera Acumulada

Nesta etapa analítica, nós transformamos os dados para quantificar o "custo de tempo" para o cliente. Primeiro, recriamos a segmentação de `faixa_etaria` para garantir a granularidade demográfica.

Em seguida, agrupamos os dados por Faixa Etária e Canal para somar todo o tempo de espera. Para tornar a análise impactante e executiva, convertemos os segundos acumulados em **Horas Totais Perdidas**. Isso permite evidenciar, por exemplo, se um grupo específico (como idosos) está sendo desproporcionalmente penalizado em canais automatizados, gerando atrito desnecessário na jornada.

In [0]:
df_sofrimento = (
    df_chamados_geral
    
    # 1. Garantia da Faixa Etária (Caso não exista na tabela origem)
    .withColumn("faixa_etaria", 
                F.when(F.col("idade") < 25, "1. Jovem (ate 24)")
                .when(F.col("idade").between(25, 40), "2. Adulto Jovem (25-40)")
                .when(F.col("idade").between(41, 60), "3. Adulto (41-60)")
                .when(F.col("idade") > 60, "4. Senior (+60)")
                .otherwise("5. Nao Informado")
    )

    # 2. Agregação de "Sofrimento"
    .groupBy("faixa_etaria", "canal") 
    .agg(
        F.count("id_chamado").alias("volumetria"),
        F.round(F.avg("tempo_espera_segundos"), 0).alias("media_espera_seg"),
        F.sum("tempo_espera_segundos").alias("soma_total_espera_seg")
    )
    
    # 3. Engenharia de Features (Transformar Segundos em Horas para chocar a gestão)
    .withColumn("horas_totais_perdidas_clientes", F.round(F.col("soma_total_espera_seg") / 3600, 2))
    .withColumn("minutos_medios_espera", F.round(F.col("media_espera_seg") / 60, 1))
    
    # Ordenação: Quem perde mais tempo aparece primeiro
    .orderBy(F.col("horas_totais_perdidas_clientes").desc())
)

display(df_sofrimento)

## Persistência do Data de Jornada de Espera

Finalizamos o processo materializando esta visão estratégica na tabela `jornada_espera` na camada Gold.

Utilizamos o formato Delta com modo de sobrescrita. Esta tabela servirá como base para justificar investimentos em transbordo de canais ou atendimento prioritário, provando matematicamente onde a "fricção" do atendimento está corroendo a lealdade dos clientes.

In [0]:
tabela_destino = f"{catalogo}.{gold_db_name}.jornada_espera"

(
    df_sofrimento.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"View de Jornada de Espera criada com sucesso: {tabela_destino}")

## Definição de Hipóteses para Digitalização

Nesta etapa, nós definimos quais são os motivos de contato considerados "simples" ou "transacionais". A premissa de negócio é que estes assuntos (como consultas de fatura, limite ou alterações cadastrais) deveriam ser resolvidos preferencialmente por canais digitais (App/Bot). Se um humano está a resolver isto, identificamos uma ineficiência operacional e financeira.

In [0]:
# Lista corrigida com os NOMES EXATOS encontrados no banco de dados
motivos_simples = [
    "Consulta De Limite",
    "Consulta De Fatura",
    "Consulta De Contrato",
    "Alteracao De Dados Cadastrais (vencimento Da Fatura, Telefone, Email)",
    "Duvidas Gerais Sobre Programa De Pontos",
    "Problema Com Aplicativo",
    "Desbloqueio De Cartao",
    "Contratacao De Cartao Adicional"
]

# Lista de canais que representam custo humano alto
canais_humanos = [
    "Atendimento Inicial", 
    "Atendimento Especializado"
]

## Cálculo de Desperdício Operacional

Para quantificar a oportunidade, nós filtramos os chamados que cumprem duas condições simultâneas: foram atendidos por um canal humano (caro) **E** trataram de um motivo simples (barato).

Como a dimensão de `faixa_etaria` não é nativa da tabela fato, recriamo-la durante o processamento para garantir a correta segmentação demográfica. Em seguida, calculamos o volume total de chamados, o custo total desperdiçado e o tempo médio gasto pela equipa humana nestas interações evitáveis.

In [0]:
from pyspark.sql import functions as F

df_potencial_digital = (
    df_chamados_geral
    
    # 1. Tratamento da Idade (Garantia contra nulos ou textos)
    .withColumn("idade_tratada", F.col("idade").cast("int"))
    
    # 2. Recriação da Faixa Etária
    .withColumn("faixa_etaria", 
                F.when(F.col("idade_tratada") < 25, "1. Jovem (ate 24)")
                .when(F.col("idade_tratada").between(25, 40), "2. Adulto Jovem (25-40)")
                .when(F.col("idade_tratada").between(41, 60), "3. Adulto (41-60)")
                .when(F.col("idade_tratada") > 60, "4. Senior (+60)")
                .otherwise("5. Nao Informado")
    )
    
    # 3. FILTRO CRÍTICO (A "Oportunidade")
    # Trazemos apenas o que foi atendido por HUMANO mas era MOTIVO SIMPLES
    .filter(
        (F.col("canal").isin(canais_humanos)) &
        (F.col("motivo").isin(motivos_simples))
    )
    
    # 4. Agregação
    .groupBy("faixa_etaria", "motivo")
    .agg(
        F.count("id_chamado").alias("volumetria_desperdicada"),
        F.round(F.sum("valor_custo"), 2).alias("custo_evitavel_total"),
        F.round(F.avg("tempo_atendimento_segundos"), 0).alias("tma_gasto_medio_seg")
    )
    
    # 5. Métricas Derivadas e Ordenação
    .withColumn("custo_medio_unitario", 
                F.round(F.col("custo_evitavel_total") / F.col("volumetria_desperdicada"), 2))
    .orderBy(F.col("custo_evitavel_total").desc())
)

display(df_potencial_digital)

## Persistência do Data de Oportunidade Digital

Finalizamos o fluxo gravando a tabela `oportunidade_digitalizacao` na camada Gold. Esta tabela consolidada servirá de base para que as equipas de Produto e Marketing direcionem campanhas de educação digital (ex: "Use o App para consultar fatura") focadas nos grupos demográficos que mais geram custos evitáveis.

In [0]:
tabela_destino = f"{catalogo}.{gold_db_name}.oportunidade_digitalizacao"

(
    df_potencial_digital.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"✅ Tabela salva com sucesso: {tabela_destino}")

## Análise de Tempo de Atendimento por Motivo


## Investigação de Eficiência Operacional por Motivo de Contato

Para compreender quais assuntos consomem mais recursos operacionais, nós agregamos os dados de atendimento por `motivo`. Esta análise permite identificar:
* **Motivos mais demorados:** Assuntos que geram TMA elevado e podem precisar de especialização.
* **Motivos mais volumosos:** Temas recorrentes que justificam investimentos em automação ou FAQs.
* **Relação Tempo x Custo:** A identificação de ineficiências onde o tempo médio não justifica a complexidade aparente.

Calculamos métricas essenciais como volumetria total, tempo médio de atendimento, tempo médio de espera e custo médio por chamado. Esta visão estratégica permite priorizar melhorias de processo e otimizações de script para os motivos críticos.

In [0]:
df_tempo_atendimento_motivo = (
    df_chamados_geral
    
    # Agregação por Motivo de Contato
    .groupBy("motivo")
    .agg(
        # Volumetria Total
        F.count("id_chamado").alias("total_chamados"),
        
        # Tempo Médio de Atendimento (TMA)
        F.round(F.avg("tempo_atendimento_segundos"), 2).alias("tma_medio_seg"),
        
        # Tempo Médio de Espera (TME)
        F.round(F.avg("tempo_espera_segundos"), 2).alias("tme_medio_seg"),
        
        # Tempo Total Acumulado (Em horas - Métrica de Impacto)
        F.round(F.sum("tempo_atendimento_segundos") / 3600, 2).alias("tempo_total_atendimento_horas"),
        
        # Custo Médio por Atendimento
        F.round(F.avg("valor_custo"), 2).alias("custo_medio_chamado"),
        
        # Custo Total Acumulado
        F.round(F.sum("valor_custo"), 2).alias("custo_total_motivo"),
        
        # Taxa de Resolução (Eficácia)
        F.round(
            (F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)) / F.count("id_chamado")) * 100, 
            2
        ).alias("taxa_resolucao_pct"),
        
        # Nota Média de Satisfação
        F.round(F.avg("nota_atendimento"), 2).alias("nota_media_satisfacao")
    )
    
    # Engenharia de Features - Conversão para Minutos (Formato mais intuitivo)
    .withColumn("tma_medio_minutos", F.round(F.col("tma_medio_seg") / 60, 2))
    .withColumn("tme_medio_minutos", F.round(F.col("tme_medio_seg") / 60, 2))
    
    # Ordenação: Motivos que mais consomem tempo aparecem primeiro
    .orderBy(F.col("tma_medio_seg").desc())
)

display(df_tempo_atendimento_motivo)

## Persistência da Análise de Tempo por Motivo

Finalizamos materializando esta visão estratégica na tabela `tempo_atendimento_motivo` na camada Gold.

Esta tabela consolidada permitirá que as equipas de Operações e Melhoria Contínua identifiquem os motivos de contato que geram maior carga operacional, possibilitando:
* Priorização de treinamentos específicos para atendentes.
* Identificação de oportunidades de automação via chatbot/FAQ.
* Justificativa para contratação de especialistas em temas complexos.
* Monitoramento da evolução do TMA ao longo do tempo após intervenções.

In [0]:
tabela_destino = f"{catalogo}.{gold_db_name}.tempo_atendimento_motivo"

(
    df_tempo_atendimento_motivo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"✅ Tabela salva com sucesso: {tabela_destino}")

## Segmentação Demográfica da Satisfação do Cliente

Para compreender se existem diferenças geracionais na percepção de qualidade do atendimento, nós cruzamos a dimensão `faixa_etaria` com a `categoria_nota` (avaliação de desempenho do atendimento).

Esta análise permite identificar:
* **Perfis mais críticos:** Faixas etárias que atribuem notas mais baixas sistematicamente.
* **Perfis promotores:** Grupos demográficos naturalmente mais satisfeitos com o serviço.
* **Oportunidades de personalização:** Adaptar a experiência de atendimento para cada geração.

Calculamos a distribuição percentual e volumétrica das categorias de avaliação (Promotor, Neutro, Detrator) dentro de cada segmento etário. Esta visão estratégica possibilita a criação de jornadas customizadas por perfil demográfico, maximizando a satisfação e reduzindo o churn.

In [0]:
df_faixa_etaria_avaliacao = (
    df_chamados_geral
    
    # 1. Criação da Faixa Etária
    .withColumn("faixa_etaria", 
                F.when(F.col("idade") < 25, "1. Jovem (ate 24)")
                .when(F.col("idade").between(25, 40), "2. Adulto Jovem (25-40)")
                .when(F.col("idade").between(41, 60), "3. Adulto (41-60)")
                .when(F.col("idade") > 60, "4. Senior (+60)")
                .otherwise("5. Nao Informado")
    )
    
    # 2. Agregação por Faixa Etária e Categoria de Nota
    .groupBy("faixa_etaria", "categoria_nota")
    .agg(
        # Volumetria Total
        F.count("id_chamado").alias("total_chamados"),
        
        # Nota Média dentro desta combinação
        F.round(F.avg("nota_atendimento"), 2).alias("nota_media"),
        
        # Tempo Médio de Atendimento (Para identificar se insatisfação está ligada ao tempo)
        F.round(F.avg("tempo_atendimento_segundos"), 2).alias("tma_medio_seg"),
        
        # Custo Médio (Para identificar se insatisfação está ligada ao custo/complexidade)
        F.round(F.avg("valor_custo"), 2).alias("custo_medio"),
        
        # Taxa de Resolução (Insatisfação pode vir de baixa resolutividade)
        F.round(
            (F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)) / F.count("id_chamado")) * 100, 
            2
        ).alias("taxa_resolucao_pct")
    )
    
    # 3. Cálculo de Percentual dentro da Faixa Etária
    .withColumn(
        "pct_dentro_faixa_etaria",
        F.round(
            (F.col("total_chamados") / F.sum("total_chamados").over(Window.partitionBy("faixa_etaria"))) * 100,
            2
        )
    )
    
    # 4. Engenharia de Features - Conversão de Tempo para Minutos
    .withColumn("tma_medio_minutos", F.round(F.col("tma_medio_seg") / 60, 2))
    
    # 5. Ordenação: Primeiro por faixa etária, depois pela categoria de nota
    .orderBy("faixa_etaria", "categoria_nota")
)

display(df_faixa_etaria_avaliacao)

## Persistência da Análise Demográfica de Satisfação

Finalizamos materializando esta visão na tabela `faixa_etaria_avaliacao` na camada Gold.

Esta tabela consolidada permitirá que as equipas de CX (Customer Experience) e Produto implementem estratégias direcionadas:
* **Ações corretivas para Detratores:** Identificar e mitigar pontos de dor específicos de cada geração.
* **Replicação de boas práticas:** Entender o que funciona bem com Promotores e expandir para outros segmentos.
* **Personalização por geração:** Adaptar canais, linguagem e processos para cada perfil etário.
* **Monitoramento de evolução:** Acompanhar se iniciativas de melhoria estão impactando positivamente cada segmento.

In [0]:
# Salvar análise demográfica de satisfação na camada Gold
tabela_destino = f"{catalogo}.{gold_db_name}.faixa_etaria_avaliacao"

save_table_gold(
    df=df_faixa_etaria_avaliacao,
    table_name="faixa_etaria_avaliacao"
)

print(f"✅ Tabela {tabela_destino} salva com sucesso!")
describe_table(df_faixa_etaria_avaliacao)